In [1]:
import pandas as pd

In [2]:
historical = pd.read_csv("MPS_Borough_Level_Crime_Historical.csv")
recent = pd.read_csv("MPS_Borough_Level_Crime_Recent_24_month.csv")

In [3]:
print("Historical shape:", historical.shape)
print("Recent shape:", recent.shape)

Historical shape: (1085, 170)
Recent shape: (1032, 27)


In [4]:
id_cols = ["MajorText", "MinorText", "BoroughName"]

historical_long = historical.melt(id_vars=id_cols, var_name="YearMonth", value_name="CrimeCount")
recent_long = recent.melt(id_vars=id_cols, var_name="YearMonth", value_name="CrimeCount")

print(historical_long.shape)
print(historical_long.head())

(181195, 5)
                   MajorText                        MinorText  \
0  ARSON AND CRIMINAL DAMAGE                            ARSON   
1  ARSON AND CRIMINAL DAMAGE                  CRIMINAL DAMAGE   
2                   BURGLARY  BURGLARY BUSINESS AND COMMUNITY   
3                   BURGLARY                DOMESTIC BURGLARY   
4                   BURGLARY           RES BURGLARY OF A HOME   

            BoroughName YearMonth  CrimeCount  
0  Barking and Dagenham    201004           6  
1  Barking and Dagenham    201004         208  
2  Barking and Dagenham    201004          49  
3  Barking and Dagenham    201004         118  
4  Barking and Dagenham    201004           0  


In [5]:
crime = pd.concat([historical_long, recent_long], ignore_index=True)

print(crime.shape)
print(crime["YearMonth"].min(), crime["YearMonth"].max())

(205963, 5)
201004 202602


In [6]:
crime["Date"] = pd.to_datetime(crime["YearMonth"], format="%Y%m")
crime["Year"] = crime["Date"].dt.year
crime["Month"] = crime["Date"].dt.month

print(crime[["YearMonth", "Date", "Year", "Month"]].head())
print(crime["Date"].dtype)

  YearMonth       Date  Year  Month
0    201004 2010-04-01  2010      4
1    201004 2010-04-01  2010      4
2    201004 2010-04-01  2010      4
3    201004 2010-04-01  2010      4
4    201004 2010-04-01  2010      4
datetime64[ns]


In [7]:
print(crime["BoroughName"].nunique())
print(sorted(crime["BoroughName"].unique()))

34
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston upon Thames', 'Lambeth', 'Lewisham', 'London Heathrow and London City Airports', 'Merton', 'Newham', 'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 'Tower Hamlets', 'Unknown', 'Waltham Forest', 'Wandsworth', 'Westminster']


In [8]:
not_boroughs = ["Unknown", "London Heathrow and London City Airports"]
crime = crime[~crime["BoroughName"].isin(not_boroughs)]

print(crime["BoroughName"].nunique())

32


In [10]:
burglary_rename = {
    "DOMESTIC BURGLARY": "RESIDENTIAL BURGLARY",
    "RES BURGLARY OF A HOME": "RESIDENTIAL BURGLARY",
    "RES BURGLARY OF UNCONNECTED BUILDING": "RESIDENTIAL BURGLARY",
    "BURGLARY - RESIDENTIAL": "RESIDENTIAL BURGLARY",
}
crime["MinorText"] = crime["MinorText"].replace(burglary_rename)

print(crime["MinorText"].unique())
print(crime["MinorText"].nunique())

['ARSON' 'CRIMINAL DAMAGE' 'BURGLARY BUSINESS AND COMMUNITY'
 'RESIDENTIAL BURGLARY' 'POSSESSION OF DRUGS' 'TRAFFICKING OF DRUGS'
 'FRAUD AND FORGERY' 'MISC CRIMES AGAINST SOCIETY' 'POSSESSION OF WEAPONS'
 'OTHER OFFENCES PUBLIC ORDER' 'PUBLIC FEAR ALARM OR DISTRESS'
 'RACE OR RELIGIOUS AGG PUBLIC FEAR' 'VIOLENT DISORDER'
 'ROBBERY OF BUSINESS PROPERTY' 'ROBBERY OF PERSONAL PROPERTY'
 'OTHER SEXUAL OFFENCES' 'RAPE' 'BICYCLE THEFT' 'OTHER THEFT'
 'SHOPLIFTING' 'THEFT FROM THE PERSON' 'AGGRAVATED VEHICLE TAKING'
 'INTERFERING WITH A MOTOR VEHICLE' 'THEFT FROM A VEHICLE'
 'THEFT OR UNAUTH TAKING OF A MOTOR VEH'
 'DEATH SERIOUS INJURY ILLEGAL DRIVING' 'HOMICIDE'
 'STALKING AND HARASSMENT' 'VIOLENCE WITH INJURY'
 'VIOLENCE WITHOUT INJURY' 'NFIB']
31


In [12]:
crime = crime.dropna(subset=["CrimeCount"])
crime = crime[crime["CrimeCount"] >= 0]


crime = crime.rename(columns={
    "BoroughName": "Borough",
    "MajorText": "MajorCategory",
    "MinorText": "MinorCategory",
})


crime = crime[["Borough", "MajorCategory", "MinorCategory", "Date", "Year", "Month", "CrimeCount"]]
crime = crime.sort_values(["Borough", "MajorCategory", "MinorCategory", "Date"])

print(crime.shape)
print(crime.head())

import os
os.makedirs("data", exist_ok=True)

crime.to_csv("data/cleaned_crime_long.csv", index=False)
print("Saved!")

(195175, 7)
                   Borough              MajorCategory MinorCategory  \
0     Barking and Dagenham  ARSON AND CRIMINAL DAMAGE         ARSON   
1085  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE         ARSON   
2170  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE         ARSON   
3255  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE         ARSON   
4340  Barking and Dagenham  ARSON AND CRIMINAL DAMAGE         ARSON   

           Date  Year  Month  CrimeCount  
0    2010-04-01  2010      4           6  
1085 2010-05-01  2010      5           5  
2170 2010-06-01  2010      6          11  
3255 2010-07-01  2010      7          10  
4340 2010-08-01  2010      8           6  
Saved!
